In [ ]:
import requests
import psycopg2
from psycopg2 import sql
# Constants
API_KEY = "59291322021a0f4c876ab24ee0c7fc3a"
USER = "gigs12345678"
URL = "http://ws.audioscrobbler.com/2.0/"
DB_CONFIG = {
    "dbname": "music_ratings",
    "user": "giovani",
    "password": "giovani",
    "host": "localhost"
}

def fetch_recent_tracks(user, api_key, limit=10):
    params = {
        "method": "user.getrecenttracks",
        "user": user,
        "api_key": api_key,
        "format": "json",
        "limit": limit
    }
    response = requests.get(URL, params=params)
    response.raise_for_status()  # Raise an error if the request fails
    return response.json()

def fetch_album_tracks(artist_name, album_name, api_key):
    params = {
        "method": "album.getinfo",
        "artist": artist_name,
        "album": album_name,
        "api_key": api_key,
        "format": "json"
    }
    try:
        response = requests.get(URL, params=params)
        # response.raise_for_status()  # Raise an error if the request fails
        return response.json()
    except:
        return None

def print_recent_tracks(data):
    print("Recent Tracks:")
    for track in data.get("recenttracks", {}).get("track", []):
        track_name = track.get("name", "Unknown")
        artist_name = track.get("artist", {}).get("#text", "Unknown")
        album_name = track.get("album", {}).get("#text", "Unknown")
        print(f"Track: {track_name}, Artist: {artist_name}, Album: {album_name}")

def parse_album_info(album_info):
    tracklist = []
    tags = []
    if "album" not in album_info.keys():
        return tracklist, tags
    if "tracks" in album_info["album"].keys():
        try:
            tracklist = [track["name"] for track in album_info["album"]["tracks"]["track"]]
        except TypeError as e:
            print(f"ERROR: {e}")
            print(album_info["album"]["tags"])
    if "tags" in album_info["album"].keys():
        try:
            tags = [tag["name"] for tag in album_info["album"]["tags"]["tag"]]
        except TypeError as e:
            print(f"ERROR: {e}")
            print(album_info["album"]["tags"])
    return tracklist, tags

def load_artist(artist_name):
    conn = get_db_connection()
    try:
        with conn.cursor() as cursor:
            query = sql.SQL("""
                INSERT INTO artists (artist_name)
                VALUES (%s)
                ON CONFLICT (artist_name) DO NOTHING;
            """)
            cursor.execute(query, (artist_name,))
        conn.commit()
    except Exception as e:
        print(f"Error inserting artist: {e}")
    finally:
        conn.close()

def load_album(artist_name, album_name):
    conn = get_db_connection()
    try:
        with conn.cursor() as cursor:
            # Step 1: Query the artist_id based on the artist_name
            query_artist = sql.SQL("""
                SELECT artist_id FROM artists WHERE artist_name = %s;
            """)
            cursor.execute(query_artist, (artist_name,))
            artist_id = cursor.fetchone()

            if artist_id:
                artist_id = artist_id[0]  # Extract the artist_id
                # Step 2: Insert the album into the albums table using artist_id
                query_album = sql.SQL("""
                    INSERT INTO albums (artist_id, album_name)
                    VALUES (%s, %s)
                    ON CONFLICT (artist_id, album_name) DO NOTHING;
                """)
                cursor.execute(query_album, (artist_id, album_name))
                conn.commit()
            else:
                print(f"Artist '{artist_name}' not found in the database.")
    except Exception as e:
        print(f"Error inserting album: {e}")
    finally:
        conn.close()

def load_track(artist_name, album_name, track_name):
    conn = get_db_connection()
    try:
        with conn.cursor() as cursor:
            # Step 1: Query the artist_id based on the artist_name
            query_artist = sql.SQL("""
                SELECT artist_id FROM artists WHERE artist_name = %s;
            """)
            cursor.execute(query_artist, (artist_name,))
            artist_id = cursor.fetchone()

            if artist_id:
                artist_id = artist_id[0]  # Extract the artist_id
                # Step 2: Query the album_id based on the artist_id and album_name
                query_album = sql.SQL("""
                    SELECT album_id FROM albums WHERE artist_id = %s AND album_name = %s;
                """)
                cursor.execute(query_album, (artist_id, album_name))
                album_id = cursor.fetchone()

                if album_id:
                    album_id = album_id[0]  # Extract the album_id
                    # Step 3: Insert the track into the tracks table using artist_id and album_id
                    query_track = sql.SQL("""
                        INSERT INTO tracks (artist_id, album_id, track_name)
                        VALUES (%s, %s, %s)
                        ON CONFLICT (artist_id, album_id, track_name) DO NOTHING;
                    """)
                    cursor.execute(query_track, (artist_id, album_id, track_name))
                    conn.commit()
                else:
                    print(f"Album '{album_name}' not found for artist '{artist_name}'.")
            else:
                print(f"Artist '{artist_name}' not found in the database.")
    except Exception as e:
        print(f"Error inserting track: {e}")
    finally:
        conn.close()
        
recent_tracks = fetch_recent_tracks(USER, API_KEY, limit=100)


In [2]:
for track in recent_tracks.get("recenttracks", {}).get("track", []):
    artist_name = track.get("artist", {}).get("#text", "Unknown")
    album_name = track.get("album", {}).get("#text", "Unknown")
    album_info = fetch_album_tracks(artist_name, album_name, API_KEY)
    tracklist, tags = parse_album_info(album_info)
    load_artist(artist_name)
    load_album(artist_name, album_name)
    for track_name in tracklist:
        load_track(artist_name, album_name, track_name)
    print(f"{artist_name} ::: {album_name} ::: {tracklist} ::: {tags}")

Racionais MC's ::: Raio X do Brasil ::: ['Fim de semana no parque', 'Parte II', 'Mano na Porta do Bar', 'Homem na Estrada', 'Júri Racional', 'Fio da Navalha', 'Voz Ativa', 'Negro Limitado', 'panico na zona sul', 'Hey Boy', 'Mulheres Vulgares', 'Racistas Otários', 'Tempos Dificeis'] ::: ['rap', 'capitalismo selvagem', 'hip hop', 'brazilian', '1994']
Racionais MC's ::: Raio X do Brasil ::: ['Fim de semana no parque', 'Parte II', 'Mano na Porta do Bar', 'Homem na Estrada', 'Júri Racional', 'Fio da Navalha', 'Voz Ativa', 'Negro Limitado', 'panico na zona sul', 'Hey Boy', 'Mulheres Vulgares', 'Racistas Otários', 'Tempos Dificeis'] ::: ['rap', 'capitalismo selvagem', 'hip hop', 'brazilian', '1994']
Racionais MC's ::: Raio X do Brasil ::: ['Fim de semana no parque', 'Parte II', 'Mano na Porta do Bar', 'Homem na Estrada', 'Júri Racional', 'Fio da Navalha', 'Voz Ativa', 'Negro Limitado', 'panico na zona sul', 'Hey Boy', 'Mulheres Vulgares', 'Racistas Otários', 'Tempos Dificeis'] ::: ['rap', 'ca

In [2]:
# Assuming fetch_all_albums is a function that fetches all albums of a given artist
def fetch_all_albums(artist_name, api_key):
    # Fetch all albums for the artist by querying the Last.fm API or using another method
    # You might use the 'artist.gettopalbums' method or something similar based on the API
    params = {
        "method": "artist.gettopalbums",
        "artist": artist_name,
        "api_key": api_key,
        "format": "json"
    }
    response = requests.get(URL, params=params)
    response.raise_for_status()
    return response.json()

artists = []
# Iterate through the recent tracks
for track in recent_tracks.get("recenttracks", {}).get("track", []):
    artist_name = track.get("artist", {}).get("#text", "Unknown")
    
    # Load the artist data
    load_artist(artist_name)
    artists.append(artist_name)

artists = set(artists)
print(artists)
for artist_name in artists:   
    # Fetch all albums by the artist
    album_data = fetch_all_albums(artist_name, API_KEY)

    # For each album of the artist, load the album and its tracks
    for album_info in album_data.get("topalbums", {}).get("album", []):
        album_name = album_info.get("name", "Unknown")
        
        # Load the album data
        load_album(artist_name, album_name)
        
        # Fetch detailed track information for the album
        album_tracks_info = fetch_album_tracks(artist_name, album_name, API_KEY)
        if album_tracks_info:
            tracklist, tags = parse_album_info(album_tracks_info)
            
            # Insert all tracks for the album
            for track_name in tracklist:
                load_track(artist_name, album_name, track_name)
        
        # Print details of the album and tracks
        print(f"{artist_name} ::: {album_name} ::: {tracklist} ::: {tags}")


{'Randy Newman', 'Smog', 'Black Country, New Road', 'The Strokes', 'Panchiko', 'Danny Brown', 'Os Originais Do Samba', 'Angela Ro Ro', 'Beto Guedes', "Flower Travellin' Band", 'Autolux', 'The Mountain Goats', 'Joelho de Porco', 'Paul McCartney', 'Rocketship', 'Tom Waits', 'Jimi Hendrix', 'Mount Eerie', 'Brian Eno', 'Neil Young', 'George Clanton', 'Novos Baianos', 'David Bowie', 'Black Alien', 'clipping.', 'Cream', 'Frances Quinlan', 'UFO', 'Jeff Rosenstock', 'Ice Cube', 'The Magnetic Fields', 'Tom Zé', 'Nirvana', 'Perfume Genius', 'Elizeth Cardoso', 'of Montreal', 'Clara Nunes', 'Black Sabbath', 'Crumb', 'Don L', 'King Gizzard & The Lizard Wizard', 'Sun Kil Moon', 'Bezerra da Silva', 'The Lemon Twigs', 'Squid', 'ZelooperZ', 'Ultimate Spinach', 'Ryu, the Runner', 'John Lennon', 'Heavenly', 'The Smile', 'Silver Jews', 'Sérgio Sampaio', 'The Gyuto Monks', 'bl4ck m4rket c4rt', 'Bic Runga', 'Radiohead', 'Thee American Revolution', 'Captain Beefheart & His Magic Band', 'Travis Scott', 'Djong